🧩 Cell 1: 安裝必要套件

In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes



🧩 Cell 2: 載入模型與 tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token



🧩 Cell 3: 套用 LoRA 組態到模型

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

from peft import get_peft_model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


/usr/local/lib/python3.11/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


🧩 Cell 4: 建立訓練資料集並加上 labels

In [ ]:
from datasets import Dataset

data = [
    {"text": "Q: 貓咪會飛嗎？ A: 不會，牠們會跳得很高。"},
    {"text": "Q: 1 + 1 等於多少？ A: 答案是 2。"}
]

dataset = Dataset.from_list(data)

def tokenize(example):
    tokens = tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)
    tokens["labels"] = tokens["input_ids"].copy()  # 加入 labels
    return tokens

tokenized_dataset = dataset.map(tokenize)



Map:   0%|          | 0/2 [00:00<?, ? examples/s]

🧩 Cell 5: 設定訓練參數與 Trainer

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./lora-tiny-output",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none"  # 關閉 wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)



No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


🧩 Cell 6: 開始訓練

In [ ]:
trainer.train()



Step,Training Loss
1,9.479500
2,9.111500
3,8.826100


TrainOutput(global_step=3, training_loss=9.139022827148438, metrics={'train_runtime': 0.749, 'train_samples_per_second': 8.011, 'train_steps_per_second': 4.005, 'total_flos': 4772223516672.0, 'train_loss': 9.139022827148438, 'epoch': 3.0})

🧩 Cell 7（可選）: 儲存 LoRA Adapter。
 - LoRA adapter 的模型參數 儲存在你目前的 Colab 工作目錄下（就是 /content/lora-tiny-output）

In [ ]:
model.save_pretrained("lora-tiny-output")


🧩 Check output file via **Terminal**
- cd /content/
- ls -l